In [1]:
%pip install torch torchvision segmentation-models-pytorch albumentations scikit-learn pandas tqdm scikit-image torchstain

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ CONFIGURATION & DIRECTORIES
# ==========================================
IMAGE_DIR = "GBM_0067_0108"
MASK_DIR = "GBM_0067_0108_GroundTruth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 10
BATCH_SIZE = 4
ACCUM_STEPS = 2
SAVE_PATH = "unet_finetuned_gbm.pth"
CSV_LOG_PATH = "patch_metrics_log_gbm.csv"

# ==========================================
# 📊 STEP 1: 80-20-10 DATA SPLITTING
# ==========================================
all_fnames = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.tif'))])

# 80% Train+Val, 20% Test
train_val_fnames, test_fnames = train_test_split(all_fnames, test_size=0.20, random_state=42)
# 10% of the 80% for Random Validation
train_fnames, val_fnames = train_test_split(train_val_fnames, test_size=0.10, random_state=42)

print(f"Splits -> Train: {len(train_fnames)}, Val: {len(val_fnames)}, Test: {len(test_fnames)}")

# ==========================================
# 🖼️ STEP 2: DATASET & METRIC FUNCTIONS
# ==========================================
class GlioblastomaDataset(Dataset):
    def __init__(self, image_dir, mask_dir, fnames, transform=None):
        self.image_dir, self.mask_dir = image_dir, mask_dir
        self.fnames = fnames
        self.transform = transform

    def __len__(self): return len(self.fnames)

    def __getitem__(self, idx):
        fn = self.fnames[idx]
        image = io.imread(os.path.join(self.image_dir, fn))
        if image.ndim == 2: image = np.stack([image]*3, axis=-1)
        if image.shape[-1] == 4: image = image[..., :3]
        
        mask = io.imread(os.path.join(self.mask_dir, fn), as_gray=True)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"].unsqueeze(0)
        return image, mask, fn

def get_patch_metrics(preds, targets, eps=1e-7):
    """Calculates Dice and Jaccard(IoU) for every individual image in the batch."""
    preds = (torch.sigmoid(preds) > 0.5).float()
    p = preds.view(preds.size(0), -1)
    t = targets.view(targets.size(0), -1)
    
    intersection = (p * t).sum(dim=1)
    total = p.sum(dim=1) + t.sum(dim=1)
    union = total - intersection
    
    dice = (2. * intersection + eps) / (total + eps)
    iou = (intersection + eps) / (union + eps) # Jaccard
    return dice.detach().cpu().numpy(), iou.detach().cpu().numpy()

# ==========================================
# 🏗️ STEP 3: MODEL & LOSS SETUP
# ==========================================
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)

class MixedLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = torch.nn.BCEWithLogitsLoss()
        self.dice = smp.losses.DiceLoss(mode="binary")
    def forward(self, p, t): return 0.4 * self.bce(p, t) + 0.6 * self.dice(p, t)

criterion = MixedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler(DEVICE)

# Transforms
train_tfms = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.Normalize(), ToTensorV2()])
val_tfms = A.Compose([A.Normalize(), ToTensorV2()])

# Loaders
train_loader = DataLoader(GlioblastomaDataset(IMAGE_DIR, MASK_DIR, train_fnames, train_tfms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(GlioblastomaDataset(IMAGE_DIR, MASK_DIR, val_fnames, val_tfms), batch_size=BATCH_SIZE)

# ==========================================
# 🚀 STEP 4: TRAINING & LOGGING LOOP
# ==========================================
patch_logs = []
best_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    print(f"\n🧠 Epoch {epoch}/{EPOCHS}")
    
    for i, (imgs, masks, fnames) in enumerate(tqdm(train_loader, desc="Training")):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        
        with torch.amp.autocast(DEVICE):
            preds = model(imgs)
            loss = criterion(preds, masks)
        
        scaler.scale(loss / ACCUM_STEPS).backward()
        
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        # Log Per Patch Metrics (Train)
        dices, ious = get_patch_metrics(preds, masks)
        for d, j, fn in zip(dices, ious, fnames):
            patch_logs.append({"epoch": epoch, "split": "train", "file": fn, "dice": d, "iou": j})

    # Validation
    model.eval()
    val_pixel_correct, val_total_pixels, epoch_val_dice, epoch_val_iou = 0, 0, [], []
    
    with torch.no_grad(), torch.amp.autocast(DEVICE):
        for imgs, masks, fnames in tqdm(val_loader, desc="Validation"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            
            # Pixel-wise Validation Accuracy
            val_pixel_correct += ((torch.sigmoid(preds) > 0.5) == masks).float().sum().item()
            val_total_pixels += torch.numel(masks)
            
            # Log Per Patch Metrics (Val)
            dices, ious = get_patch_metrics(preds, masks)
            epoch_val_dice.extend(dices)
            epoch_val_iou.extend(ious)
            for d, j, fn in zip(dices, ious, fnames):
                patch_logs.append({"epoch": epoch, "split": "val", "file": fn, "dice": d, "iou": j})

    # Summarize Epoch
    avg_val_acc = (val_pixel_correct / val_total_pixels) * 100
    avg_val_dice = np.mean(epoch_val_dice)
    avg_val_iou = np.mean(epoch_val_iou)
    print(f"📊 Val Accuracy: {avg_val_acc:.2f}% | Val Dice: {avg_val_dice:.4f} | Val IoU: {avg_val_iou:.4f}")

    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"💾 Saved Best Model!")

# ==========================================
# 📁 STEP 5: EXPORT CSV
# ==========================================
pd.DataFrame(patch_logs).to_csv(CSV_LOG_PATH, index=False)
print(f"\n✅ All results saved to {CSV_LOG_PATH}")

Splits -> Train: 71067, Val: 7897, Test: 19741

🧠 Epoch 1/1


Training:  10%|█         | 1840/17767 [01:34<13:37, 19.49it/s]


KeyboardInterrupt: 

In [3]:
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
import torchstain
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ CONFIGURATION & DIRECTORIES
# ==========================================
IMAGE_DIR = "GBM_0067_0108"
MASK_DIR = "GBM_0067_0108_GroundTruth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 1
BATCH_SIZE = 4
ACCUM_STEPS = 2
MODEL_SAVE_PATH = "unet_finetuned_best-2.pth"
TRAIN_LOG_CSV = "train_patch_metrics.csv"
TEST_LOG_CSV = "test_patch_metrics.csv"
PRED_OUT_DIR = "test_predictions"

os.makedirs(PRED_OUT_DIR, exist_ok=True)

# ==========================================
# 📊 STEP 1: 80-20-10 DATA SPLITTING
# ==========================================
all_fnames = sorted([f for f in os.listdir(IMAGE_DIR) if f.endswith(('.png', '.jpg', '.tif'))])
train_val_fnames, test_fnames = train_test_split(all_fnames, test_size=0.20, random_state=42)
train_fnames, val_fnames = train_test_split(train_val_fnames, test_size=0.10, random_state=42)

print(f"Splits -> Train: {len(train_fnames)}, Val: {len(val_fnames)}, Test: {len(test_fnames)}")

# ==========================================
# 🧪 STEP 2: STAIN NORMALIZATION SETUP (REINHARD)
# ==========================================
# Use the first training image as the color reference
target_sample = io.imread(os.path.join(IMAGE_DIR, train_fnames[0]))
if target_sample.shape[-1] == 4: target_sample = target_sample[..., :3]

normalizer = torchstain.normalizers.ReinhardNormalizer(backend='numpy')
normalizer.fit(target_sample)

# ==========================================
# 🖼️ STEP 3: DATASET & METRICS
# ==========================================
class GlioblastomaDataset(Dataset):
    def __init__(self, image_dir, mask_dir, fnames, normalizer=None, transform=None):
        self.image_dir, self.mask_dir = image_dir, mask_dir
        self.fnames = fnames
        self.transform = transform
        self.normalizer = normalizer

    def __len__(self): return len(self.fnames)

    def __getitem__(self, idx):
        fn = self.fnames[idx]
        image = io.imread(os.path.join(self.image_dir, fn))
        if image.ndim == 2: image = np.stack([image]*3, axis=-1)
        if image.shape[-1] == 4: image = image[..., :3]
        
        # Apply Reinhard Normalization
        if self.normalizer:
            try:
                image = self.normalizer.normalize(image)
                image = np.clip(image, 0, 255).astype(np.uint8)
            except: pass 

        mask = io.imread(os.path.join(self.mask_dir, fn), as_gray=True)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"].unsqueeze(0)
        return image, mask, fn

def get_patch_metrics(preds, targets, eps=1e-7):
    preds = (torch.sigmoid(preds) > 0.5).float()
    p, t = preds.view(preds.size(0), -1), targets.view(targets.size(0), -1)
    intersection = (p * t).sum(dim=1)
    total = p.sum(dim=1) + t.sum(dim=1)
    dice = (2. * intersection + eps) / (total + eps)
    iou = (intersection + eps) / (total - intersection + eps) # Jaccard
    return dice.detach().cpu().numpy(), iou.detach().cpu().numpy()

# ==========================================
# 🏗️ STEP 4: MODEL, LOSS, & LOADERS
# ==========================================
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)

class MixedLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.bce, self.dice = torch.nn.BCEWithLogitsLoss(), smp.losses.DiceLoss(mode="binary")
    def forward(self, p, t): return 0.4 * self.bce(p, t) + 0.6 * self.dice(p, t)

criterion = MixedLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler(DEVICE)

train_tfms = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.Normalize(), ToTensorV2()])
val_tfms = A.Compose([A.Normalize(), ToTensorV2()])

train_loader = DataLoader(GlioblastomaDataset(IMAGE_DIR, MASK_DIR, train_fnames, normalizer, train_tfms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(GlioblastomaDataset(IMAGE_DIR, MASK_DIR, val_fnames, normalizer, val_tfms), batch_size=BATCH_SIZE)
test_loader = DataLoader(GlioblastomaDataset(IMAGE_DIR, MASK_DIR, test_fnames, normalizer, val_tfms), batch_size=1)

# ==========================================
# 🚀 STEP 5: TRAINING LOOP
# ==========================================
patch_logs = []
best_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    print(f"\n🧠 Epoch {epoch}/{EPOCHS}")
    for i, (imgs, masks, fnames) in enumerate(tqdm(train_loader, desc="Training")):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with torch.amp.autocast(DEVICE):
            preds = model(imgs)
            loss = criterion(preds, masks)
        scaler.scale(loss / ACCUM_STEPS).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        d, j = get_patch_metrics(preds, masks)
        for val_d, val_j, fn in zip(d, j, fnames):
            patch_logs.append({"epoch": epoch, "split": "train", "file": fn, "dice": val_d, "jaccard": val_j})

    # Validation Accuracy
    model.eval()
    val_pixel_correct, val_total_pixels, epoch_val_dice = 0, 0, []
    with torch.no_grad(), torch.amp.autocast(DEVICE):
        for imgs, masks, fnames in tqdm(val_loader, desc="Validation"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            val_pixel_correct += ((torch.sigmoid(preds) > 0.5) == masks).float().sum().item()
            val_total_pixels += torch.numel(masks)
            d, j = get_patch_metrics(preds, masks)
            epoch_val_dice.extend(d)
            for val_d, val_j, fn in zip(d, j, fnames):
                patch_logs.append({"epoch": epoch, "split": "val", "file": fn, "dice": val_d, "jaccard": val_j})

    avg_val_acc = (val_pixel_correct / val_total_pixels) * 100
    avg_val_dice = np.mean(epoch_val_dice)
    print(f"📊 Val Accuracy: {avg_val_acc:.2f}% | Val Dice: {avg_val_dice:.4f}")

    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("💾 Saved Best Model!")

pd.DataFrame(patch_logs).to_csv(TRAIN_LOG_CSV, index=False)

# ==========================================
# 🔍 STEP 6: INFERENCE (TEST SET)
# ==========================================
print("\n🚀 Starting Inference on Test Set...")
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
test_logs = []

with torch.no_grad(), torch.amp.autocast(DEVICE):
    for imgs, masks, fnames in tqdm(test_loader, desc="Testing"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        d, j = get_patch_metrics(preds, masks)
        
        # Save Binary Mask
        mask_np = (torch.sigmoid(preds)[0,0] > 0.5).cpu().numpy().astype(np.uint8) * 255
        io.imsave(os.path.join(PRED_OUT_DIR, fnames[0]), mask_np, check_contrast=False)
        
        test_logs.append({"file": fnames[0], "dice": d[0], "jaccard": j[0]})

pd.DataFrame(test_logs).to_csv(TEST_LOG_CSV, index=False)
print(f"✅ Done! Test Metrics: {pd.DataFrame(test_logs)['dice'].mean():.4f} Dice")

Splits -> Train: 71067, Val: 7897, Test: 19741

🧠 Epoch 1/1


Validation: 100%|██████████| 1975/1975 [02:29<00:00, 13.18it/s]


📊 Val Accuracy: 96.26% | Val Dice: 0.7941
💾 Saved Best Model!

🚀 Starting Inference on Test Set...


Testing: 100%|██████████| 19741/19741 [11:32<00:00, 28.52it/s]


✅ Done! Test Metrics: 0.7912 Dice


In [2]:
import os
import torch
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
from skimage import io
import torchstain
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# ==========================================
# ⚙️ CONFIGURATION & DIRECTORIES
# ==========================================
SOURCE_IMG = "GBM_0067_0108"
SOURCE_MASK = "GBM_0067_0108_GroundTruth"
BASE_SPLIT_DIR = "Glioblastoma_Splits"  # New root for split folders

# Create physical directories
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(BASE_SPLIT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(BASE_SPLIT_DIR, split, "masks"), exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 10
BATCH_SIZE = 4
ACCUM_STEPS = 2
MODEL_SAVE_PATH = "unet_finetuned_gbm.pth"
TRAIN_LOG_CSV = "train_patch_metrics_gbm.csv"
TEST_LOG_CSV = "test_patch_metrics_gbm.csv"
PRED_OUT_DIR = "test_predictions_gbm"

os.makedirs(PRED_OUT_DIR, exist_ok=True)

# ==========================================
# 📊 STEP 1: PHYSICAL DATA SPLITTING (80-20-10)
# ==========================================
all_fnames = sorted([f for f in os.listdir(SOURCE_IMG) if f.endswith(('.png', '.jpg', '.tif'))])
train_val_fnames, test_fns = train_test_split(all_fnames, test_size=0.20, random_state=42)
train_fns, val_fns = train_test_split(train_val_fnames, test_size=0.10, random_state=42)

def copy_to_folders(files, split_name):
    for f in files:
        shutil.copy(os.path.join(SOURCE_IMG, f), os.path.join(BASE_SPLIT_DIR, split_name, "images", f))
        shutil.copy(os.path.join(SOURCE_MASK, f), os.path.join(BASE_SPLIT_DIR, split_name, "masks", f))

print("🚚 Physically splitting images into folders...")
copy_to_folders(train_fns, 'train')
copy_to_folders(val_fns, 'val')
copy_to_folders(test_fns, 'test')

# ==========================================
# 🧪 STEP 2: STAIN NORMALIZATION SETUP
# ==========================================
target_sample = io.imread(os.path.join(BASE_SPLIT_DIR, "train/images", train_fns[0]))
if target_sample.shape[-1] == 4: target_sample = target_sample[..., :3]

normalizer = torchstain.normalizers.ReinhardNormalizer(backend='numpy')
normalizer.fit(target_sample)

# ==========================================
# 🖼️ STEP 3: DATASET & METRIC FUNCTIONS
# ==========================================
class GlioblastomaDataset(Dataset):
    def __init__(self, split_name, normalizer=None, transform=None):
        self.image_dir = os.path.join(BASE_SPLIT_DIR, split_name, "images")
        self.mask_dir = os.path.join(BASE_SPLIT_DIR, split_name, "masks")
        self.fnames = sorted(os.listdir(self.image_dir))
        self.transform = transform
        self.normalizer = normalizer

    def __len__(self): return len(self.fnames)

    def __getitem__(self, idx):
        fn = self.fnames[idx]
        image = io.imread(os.path.join(self.image_dir, fn))
        if image.ndim == 2: image = np.stack([image]*3, axis=-1)
        if image.shape[-1] == 4: image = image[..., :3]
        
        if self.normalizer:
            try:
                image = self.normalizer.normalize(image)
                image = np.clip(image, 0, 255).astype(np.uint8)
            except: pass 

        mask = io.imread(os.path.join(self.mask_dir, fn), as_gray=True)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image, mask = aug["image"], aug["mask"].unsqueeze(0)
        return image, mask, fn

def get_patch_metrics(preds, targets, eps=1e-7):
    preds = (torch.sigmoid(preds) > 0.5).float()
    p, t = preds.view(preds.size(0), -1), targets.view(targets.size(0), -1)
    intersection = (p * t).sum(dim=1)
    total = p.sum(dim=1) + t.sum(dim=1)
    union = total - intersection
    dice = (2. * intersection + eps) / (total + eps)
    iou = (intersection + eps) / (union + eps)
    return dice.detach().cpu().numpy(), iou.detach().cpu().numpy()

# ==========================================
# 🏗️ STEP 4: MODEL & LOADERS
# ==========================================
model = smp.Unet("resnet34", encoder_weights="imagenet", in_channels=3, classes=1).to(DEVICE)
criterion = smp.losses.DiceLoss(mode="binary") # You can swap back to MixedLoss if needed
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler(DEVICE)

train_tfms = A.Compose([A.HorizontalFlip(p=0.5), A.Normalize(), ToTensorV2()])
val_tfms = A.Compose([A.Normalize(), ToTensorV2()])

train_loader = DataLoader(GlioblastomaDataset('train', normalizer, train_tfms), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(GlioblastomaDataset('val', normalizer, val_tfms), batch_size=BATCH_SIZE)
test_loader = DataLoader(GlioblastomaDataset('test', normalizer, val_tfms), batch_size=1)

# ==========================================
# 🚀 STEP 5: TRAINING & VALIDATION
# ==========================================
patch_logs = []
best_dice = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    print(f"\n🧠 Epoch {epoch}/{EPOCHS}")
    for i, (imgs, masks, fnames) in enumerate(tqdm(train_loader, desc="Training")):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with torch.amp.autocast(DEVICE):
            preds = model(imgs)
            loss = criterion(preds, masks)
        scaler.scale(loss / ACCUM_STEPS).backward()
        if (i + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)

        d, j = get_patch_metrics(preds, masks)
        for val_d, val_j, fn in zip(d, j, fnames):
            patch_logs.append({"epoch": epoch, "split": "train", "file": fn, "dice": val_d, "iou": val_j})

    model.eval()
    val_pixel_correct, val_total_pixels, epoch_val_dice, epoch_val_iou = 0, 0, [], []
    with torch.no_grad(), torch.amp.autocast(DEVICE):
        for imgs, masks, fnames in tqdm(val_loader, desc="Validation"):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            val_pixel_correct += ((torch.sigmoid(preds) > 0.5) == masks).float().sum().item()
            val_total_pixels += torch.numel(masks)
            d, j = get_patch_metrics(preds, masks)
            epoch_val_dice.extend(d); epoch_val_iou.extend(j)
            for val_d, val_j, fn in zip(d, j, fnames):
                patch_logs.append({"epoch": epoch, "split": "val", "file": fn, "dice": val_d, "iou": val_j})

    avg_val_acc = (val_pixel_correct / val_total_pixels) * 100
    avg_val_dice, avg_val_iou = np.mean(epoch_val_dice), np.mean(epoch_val_iou)
    print(f"📊 Val Accuracy: {avg_val_acc:.2f}% | Val Dice: {avg_val_dice:.4f} | Val IoU: {avg_val_iou:.4f}")

    if avg_val_dice > best_dice:
        best_dice = avg_val_dice
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print("💾 Saved Best Model!")

pd.DataFrame(patch_logs).to_csv(TRAIN_LOG_CSV, index=False)

# ==========================================
# 🔍 STEP 6: INFERENCE (TEST SET)
# ==========================================
print("\n🚀 Starting Inference on Test Set...")
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()
test_logs = []

with torch.no_grad():
    for imgs, masks, fnames in tqdm(test_loader, desc="Testing"):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        preds = model(imgs)
        d, j = get_patch_metrics(preds, masks)
        
        mask_np = (torch.sigmoid(preds)[0,0] > 0.5).cpu().numpy().astype(np.uint8) * 255
        io.imsave(os.path.join(PRED_OUT_DIR, fnames[0]), mask_np, check_contrast=False)
        test_logs.append({"file": fnames[0], "dice": d[0], "iou": j[0]})

pd.DataFrame(test_logs).to_csv(TEST_LOG_CSV, index=False)
print(f"✅ Final Test Metrics -> Mean Dice: {pd.DataFrame(test_logs)['dice'].mean():.4f} | Mean IoU: {pd.DataFrame(test_logs)['iou'].mean():.4f}")

/home/pathouser1/.cellpose/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚚 Physically splitting images into folders...

🧠 Epoch 1/10


Validation: 100%|██████████| 1975/1975 [02:34<00:00, 12.80it/s]


📊 Val Accuracy: 96.44% | Val Dice: 0.7973 | Val IoU: 0.7191
💾 Saved Best Model!

🧠 Epoch 2/10


Validation: 100%|██████████| 1975/1975 [02:37<00:00, 12.51it/s]


...

📊 Val Accuracy: 96.46% | Val Dice: 0.8147 | Val IoU: 0.7452

🧠 Epoch 10/10


Validation: 100%|██████████| 1975/1975 [02:31<00:00, 13.02it/s]


📊 Val Accuracy: 96.33% | Val Dice: 0.8122 | Val IoU: 0.7463

🚀 Starting Inference on Test Set...


Testing: 100%|██████████| 19741/19741 [07:40<00:00, 42.82it/s]


✅ Final Test Metrics -> Mean Dice: 0.8165 | Mean IoU: 0.7456
